Initial training was performed with the SwissProt Training/Validation dataset used to train the DeepLoc 2.0 server. When the model failed to generalise well to the DeepLoc 2.0 Human Protein Atlas (HPA) test set, a subset of this dataset was used for the fine tuning step of transfer learning, to improve the model's performance across a mixture of eukaryotic species and human-only datasets. To avoid training the model based on testing results, a better future approach would be to use training, cross-validation and test datasets that are more balanced across different eukaryotic species. All data used is available at: https://services.healthtech.dtu.dk/services/DeepLoc-2.0/4-Data.php

**References**

Thumuluri, V., Almagro Armenteros, J. J., Johansen, A. R., Nielsen, H. & Winther, O. (2022). *DeepLoc 2.0: multi-label subcellular localization prediction using protein language models.* Nucleic Acids Research, 50(W1), W228–W234. https://doi.org/10.1093/nar/gkac278

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras import layers, models
from tensorflow.keras.losses import BinaryCrossentropy

In [2]:
df = pd.read_csv("Swissprot_Train_Validation_dataset.csv")
df.head()

,Unnamed: 0,ACC,Kingdom,Partition,Membrane,Cytoplasm,Nucleus,Extracellular,Cell membrane,Mitochondrion,Plastid,Endoplasmic reticulum,Lysosome/Vacuole,Golgi apparatus,Peroxisome,Sequence
0,0,Q28165,Metazoa,4,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
1,1,Q86U42,Metazoa,4,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
2,2,Q0GA42,Metazoa,3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAALGVRLRDCCSRGAVLLLFFSLSPRPPAAAAWLLGLR...
3,3,P82349,Metazoa,1,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAATEQQGSNGPVKKSMREKAVERRNVNKEHNSNFKAGY...
4,4,Q7L5N1,Metazoa,1,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAATNGTGGSSGMEVDAAVVPSVMACGVTGSVSVALHPL...


In [3]:
df.shape

(28303, 16)

In [4]:
df = df.dropna(subset = ["Partition", "Sequence", "Nucleus"])
df.shape

(28303, 16)

In [5]:
df = df[["Partition", "Nucleus", "Sequence"]]
df.head()

,Partition,Nucleus,Sequence
0,4,1.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
1,4,1.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
2,3,0.0,MAAAAAAAAALGVRLRDCCSRGAVLLLFFSLSPRPPAAAAWLLGLR...
3,1,0.0,MAAAAAAAAATEQQGSNGPVKKSMREKAVERRNVNKEHNSNFKAGY...
4,1,1.0,MAAAAAAAAATNGTGGSSGMEVDAAVVPSVMACGVTGSVSVALHPL...


In [8]:
#subsetting for maximum length of 1000 amino acids to reduce inefficiency of one-hot encoding, with the disadvantage of losing training data on the longest proteins
df = df[df["Sequence"].str.len() <= 1000]
df.shape

(25117, 3)

In [9]:
df_test = pd.read_csv("hpa_testset.csv")
df_test.head()

,sid,Cell membrane,Cytoplasm,Endoplasmic reticulum,Golgi apparatus,Lysosome/Vacuole,Mitochondrion,Nucleus,Peroxisome,Lengths,fasta
0,ENSP00000355743,0,0,0,0,0,0,1,0,496,MAAAAGPGAALSPRPCDSDPATPGAQSPKDDNEDNSNDGTQPSKRR...
1,ENSP00000429628,0,0,0,0,0,0,1,0,173,MAAAALRDPAQVPVAADLLTDHEEGYVTFEDVAVYFSQEEWRLLDD...
2,ENSP00000266544,0,0,0,0,0,1,0,0,377,MAAAAQSRVVRVLSMSRSAITAIATSVCHGPPCRQLHHALMPHGKG...
3,ENSP00000470652,0,0,0,1,0,0,0,0,410,MAAAAVGAGHGAGGPGAASSSGGAREGARVAALCLLWYALSAGGNV...
4,ENSP00000460751,0,0,0,0,0,0,1,0,163,MAAAAVTRGTPGENSHHLKIFLPKKLLECLPRCPLLPPERLRWNTN...


In [10]:
df_test.shape

(1717, 11)

In [11]:
df_test.rename(columns={"fasta": "Sequence"}, inplace=True)
df_test = df_test.dropna(subset = ["Sequence", "Nucleus"])

In [12]:
df_test = df_test[["Nucleus", "Sequence"]]
df_test.head()

,Nucleus,Sequence
0,1,MAAAAGPGAALSPRPCDSDPATPGAQSPKDDNEDNSNDGTQPSKRR...
1,1,MAAAALRDPAQVPVAADLLTDHEEGYVTFEDVAVYFSQEEWRLLDD...
2,0,MAAAAQSRVVRVLSMSRSAITAIATSVCHGPPCRQLHHALMPHGKG...
3,0,MAAAAVGAGHGAGGPGAASSSGGAREGARVAALCLLWYALSAGGNV...
4,1,MAAAAVTRGTPGENSHHLKIFLPKKLLECLPRCPLLPPERLRWNTN...


In [14]:
df_test = df_test[df_test["Sequence"].str.len() <= 1000]
df_test.shape

(1644, 2)

In [15]:
#checking distribution of lengths of proteins after dropping longest ones
df["Sequence"].str.len().describe()

count    25117.000000
mean       424.769121
std        228.721412
min         40.000000
25%        243.000000
50%        393.000000
75%        575.000000
max       1000.000000
Name: Sequence, dtype: float64

In [16]:
df_test["Sequence"].str.len().describe()

count    1644.000000
mean      234.736618
std       196.015717
min        23.000000
25%        95.000000
50%       164.000000
75%       307.000000
max       992.000000
Name: Sequence, dtype: float64

In [27]:
#feature engineering - one-hot encoding amino acids in every position
#significantly more positional information preserved compared to amino acid composition and dipeptide composition as features for simpler models
amino_acids = "ACDEFGHIKLMNPQRSTVWY"

aa_to_idx = {aa: i for i, aa in enumerate(amino_acids)}

def one_hot_encode(sequence, max_length):
    encoded = np.zeros((max_length, 20))
    for i, aa in enumerate(sequence[:max_length]):
        if aa in aa_to_idx:
            encoded[i, aa_to_idx[aa]] = 1.0
    return encoded

In [51]:
#use DeepLoc 2.0 dataset's homology partitions to split training set
train_df = df[df["Partition"].isin([0, 1, 2, 3])]
max_X_train = train_df["Sequence"].str.len().max()
X_train_seq = np.stack(train_df["Sequence"].values)
y_train = np.stack(train_df["Nucleus"].values)

cv_df = df[df["Partition"] == 4]
max_X_cv = cv_df["Sequence"].str.len().max()
X_cv_seq = np.stack(cv_df["Sequence"].values)
y_cv = np.stack(cv_df["Nucleus"].values)

max_X_test = df_test["Sequence"].str.len().max()
X_test_seq = np.stack(df_test["Sequence"].values)
y_test = np.stack(df_test["Nucleus"].values)

In [55]:
#one-hot encode features
X_train = np.array([one_hot_encode(sequence, max_X_train) for sequence in X_train_seq])
X_cv = np.array([one_hot_encode(sequence, max_X_cv) for sequence in X_cv_seq])
X_test = np.array([one_hot_encode(sequence, max_X_test) for sequence in X_test_seq])

In [79]:
model = models.Sequential([
    layers.Conv1D(
        filters=64,
        kernel_size=5,
        activation="relu"
    ),
    layers.MaxPooling1D(pool_size=2),

    # Second convolution
    layers.Conv1D(
        filters=128,
        kernel_size=5,
        activation="relu"
    ),

    # Collapse sequence dimension
    layers.GlobalMaxPooling1D(),

    layers.Dense(
        64,
        activation="relu"
    ),

    # Binary output
    layers.Dense(
        1,
        activation="sigmoid"
    )
])

model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.001),loss = BinaryCrossentropy())

model.fit(X_train, y_train, epochs = 20, batch_size = 128)

Epoch 1/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 57s 321ms/step - loss: 0.5840
Epoch 2/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 37s 234ms/step - loss: 0.4854
Epoch 3/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 34s 215ms/step - loss: 0.4668
Epoch 4/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 34s 213ms/step - loss: 0.4437
Epoch 5/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 35s 218ms/step - loss: 0.4203
Epoch 6/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 35s 223ms/step - loss: 0.4043
Epoch 7/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 37s 233ms/step - loss: 0.3746
Epoch 8/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 38s 239ms/step - loss: 0.3324
Epoch 9/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 39s 247ms/step - loss: 0.2788
Epoch 10/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 42s 262ms/step - loss: 0.2409
Epoch 11/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 40s 250ms/step - loss: 0.1948
Epoch 12/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 43s 271ms/step - loss: 0.1651
Epoch 13/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 48s 305ms/step - loss: 0.1127
Epoch 14/20
159/159 ━━━━━━━━━━━━━━━━━━━━ 40s 249ms/step - loss: 0.0882
Epoch 15/20
159

In [80]:
y_prob = model.predict(X_train)
threshold = 0.50
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (training set):", accuracy_score(y_train, yhat))
print(classification_report(y_train, yhat))
print(confusion_matrix(y_train, yhat))

635/635 ━━━━━━━━━━━━━━━━━━━━ 13s 19ms/step
Accuracy (training set): 0.9975379160921805
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     13535
         1.0       1.00      1.00      1.00      6773

    accuracy                           1.00     20308
   macro avg       1.00      1.00      1.00     20308
weighted avg       1.00      1.00      1.00     20308

[[13506    29]
 [   21  6752]]


In [81]:
y_prob = model.predict(X_test)
threshold = 0.50
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (test set):", accuracy_score(y_test, yhat))
print(classification_report(y_test, yhat))
print(confusion_matrix(y_test, yhat))

52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step
Accuracy (test set): 0.6295620437956204
              precision    recall  f1-score   support

           0       0.60      0.67      0.63       788
           1       0.66      0.59      0.62       856

    accuracy                           0.63      1644
   macro avg       0.63      0.63      0.63      1644
weighted avg       0.63      0.63      0.63      1644

[[528 260]
 [349 507]]


In [82]:
#model generalised badly to human-only protein sequences
#transfer learning to fine tune for human protein prediction
from sklearn.model_selection import train_test_split

X_hpa_train, X_hpa_test, y_hpa_train, y_hpa_test = train_test_split(X_test, y_test, test_size = 0.30, random_state =1)

In [89]:
model.fit(X_hpa_train, y_hpa_train, epochs = 10, batch_size = 32)

Epoch 1/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 82ms/step - loss: 1.2597
Epoch 2/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - loss: 0.3504
Epoch 3/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - loss: 0.1913
Epoch 4/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 83ms/step - loss: 0.0797
Epoch 5/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 82ms/step - loss: 0.0358
Epoch 6/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 81ms/step - loss: 0.0208
Epoch 7/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - loss: 0.0133
Epoch 8/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 78ms/step - loss: 0.0095
Epoch 9/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 82ms/step - loss: 0.0070
Epoch 10/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 83ms/step - loss: 0.0055


In [90]:
#test on part of HPA dataset not used for training at any point
y_prob = model.predict(X_hpa_test)
threshold = 0.50
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (test set - HPA):", accuracy_score(y_hpa_test, yhat))
print(classification_report(y_hpa_test, yhat))
print(confusion_matrix(y_hpa_test, yhat))

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
Accuracy (test set - HPA): 0.7044534412955465
              precision    recall  f1-score   support

           0       0.67      0.73      0.70       231
           1       0.74      0.68      0.71       263

    accuracy                           0.70       494
   macro avg       0.71      0.71      0.70       494
weighted avg       0.71      0.70      0.70       494

[[168  63]
 [ 83 180]]


In [91]:
#check whether model now generalises to the initial cross-validation set of eukaryotic proteins (animals, plants, fungi, etc.)
y_prob = model.predict(X_cv)
threshold = 0.50
yhat = (y_prob >= threshold).astype(int).ravel()

print("Accuracy (test set - eukaryotic):", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

151/151 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step
Accuracy (test set - eukaryotic): 0.6951549178623414
              precision    recall  f1-score   support

         0.0       0.82      0.67      0.74      3050
         1.0       0.56      0.74      0.64      1759

    accuracy                           0.70      4809
   macro avg       0.69      0.70      0.69      4809
weighted avg       0.72      0.70      0.70      4809

[[2044 1006]
 [ 460 1299]]
